# Move selection: the teacher policy ("constructive smurfing")

**Milestones:** rating-targeted play (~1700), fallback behaviour, keep games close (README
Teaching; design goals "keep games close" and "punish habits, not playstyles").

Given a position, the teacher:

1. **Checks the player model.** If confidence < `min_confidence`, or the current game is out of
   distribution (`ood_score > max_ood_score`), it **falls back** to solid, strength-limited play at
   the last estimated level (README: *"plays solid, strength-limited moves at its last estimated
   level and does not try to exploit a habit"*).
2. Otherwise gets **candidates** from the analyst (`multipv`) and keeps only those that give up at
   most `max_eval_drop_cp` versus the best move and never walk into a forced mate.
3. **Scores** each candidate: `w_close * closeness + w_target * weakness_target`, where closeness
   prefers evals inside `+-closeness_band_cp` and `weakness_target` rewards moves that set
   the player a problem their profile says they get wrong (e.g. a position where the natural reply
   is a capture they tend to miss).

This class implements `MovePolicy.choose_move(board)` from `gambit_game/notebooks/game_manager.ipynb`,
so it plugs straight into the game manager.

**TODO checklist**
- [ ] Candidate band filter and closeness score.
- [ ] Fallback logic, scoring, decision record (`TeacherPolicy`).
- [ ] Self-checks (plan at the bottom).
- [ ] `weakness_target_score` heuristics (one per weakness in `PlayerModel.weaknesses()`).
- [ ] Weights `w_close` / `w_target`; maybe reveal more strength as the player improves (README).
- [ ] Open question: should Gambit ever play at full strength to avoid losing? (disclosure.ipynb)

In [ ]:
%run ./player_model.ipynb

In [ ]:
import math
from dataclasses import dataclass, field
from typing import Callable, Optional

import chess

## Decision record (logged with every robot move so experiments can be explained later)

In [ ]:
FALLBACK_LOW_CONFIDENCE = "fallback_low_confidence"
FALLBACK_OOD = "fallback_ood"
FALLBACK_NO_CANDIDATES = "fallback_no_candidates"
TEACHING = "teaching"


@dataclass
class TeacherDecision:
    move: chess.Move
    reason: str
    confidence: float
    ood_score: float
    candidates: list = field(default_factory=list)       # Candidates considered
    scores: dict = field(default_factory=dict)           # uci -> score

    @property
    def is_fallback(self) -> bool:
        """Any reason other than TEACHING."""
        # TODO: implement
        raise NotImplementedError

## Candidate filtering and closeness

In [ ]:
def band_filter(cands: list[Candidate], max_drop: int = TEACH_CFG["max_eval_drop_cp"]) -> list[Candidate]:
    """Keep candidates within `max_drop` of the best and not losing to a forced mate."""
    # Design: compare each score with the best candidate's score (side-to-move POV, so higher is
    # better for the robot); drop anything with a negative mate distance (we get mated).
    # TODO: implement
    raise NotImplementedError


def closeness_score(score_cp: int, band: int = TEACH_CFG["closeness_band_cp"]) -> float:
    """1 inside the band around 0, decreasing linearly to 0 at 3x the band (robot POV)."""
    # Design: symmetric in the sign (being far ahead is as "not close" as being far behind).
    # TODO: implement
    raise NotImplementedError


def weakness_target_score(board: chess.Board, cand: Candidate, estimate: PlayerEstimate) -> float:
    """How well `cand` tests one of the player's weaknesses, in 0..1."""
    # Design hints, for each Weakness (ordered by `low` bound, i.e. most confident first):
    # - missed_forcing: after `cand`, does the player's best reply exist and is it a capture/check
    #   that is easy to overlook (e.g. a quiet-looking capture, a backward capture)?
    # - blunder: does `cand` create a position where many natural moves lose material (count
    #   legal replies whose eval drops > blunder threshold, via a quick analyst call)?
    # - phase weakness: steer toward the phase with the highest ACPL (trade into endgames, etc.).
    # Punish habits, not playstyles: never score a move higher just because it is aggressive.
    # TODO: implement
    raise NotImplementedError

## The policy

In [ ]:
class TeacherPolicy:
    def __init__(self, player_engine: EngineLike, analyst: EngineLike, model: PlayerModel,
                 cfg: dict = TEACH_CFG, multipv: int = ENGINE_CFG["multipv"],
                 target_fn: Callable = weakness_target_score, w_close: float = 1.0, w_target: float = 1.0,
                 min_live_moves: int = 10):
        self.player_engine, self.analyst, self.model = player_engine, analyst, model
        self.cfg, self.multipv, self.target_fn = cfg, multipv, target_fn
        self.w_close, self.w_target = w_close, w_target
        self.min_live_moves = min_live_moves
        self.live_analyses: list[MoveAnalysis] = []     # the human's moves in the current game
        self.decisions: list[TeacherDecision] = []

    def observe_human_move(self, analysis: MoveAnalysis) -> None:
        """Feed each analysed human move during the game (for the live OOD check)."""
        # TODO: implement
        raise NotImplementedError

    def live_ood_score(self) -> float:
        """OOD score of the current game so far vs the model's history."""
        # Design: return 0 (= "looks normal") until min_live_moves human moves are in, because a
        # handful of moves cannot be judged; then the model's ood_score on the live analyses.
        # TODO: implement
        raise NotImplementedError

    def solid_move(self, board: chess.Board, estimate: Optional[PlayerEstimate]) -> chess.Move:
        """Strength-limited engine move at the last estimated level (or the default Elo)."""
        # Design: if the player engine supports set_strength, set it to the estimate's Elo (default
        # Elo when there is no history), then play. Fallback engines without set_strength just play.
        # TODO: implement
        raise NotImplementedError

    def decide(self, board: chess.Board) -> TeacherDecision:
        """Full decision for one position (see the numbered steps at the top)."""
        # Design:
        # - Estimate (None without history), confidence, live OOD score.
        # - Fallback order: low confidence -> FALLBACK_LOW_CONFIDENCE; OOD -> FALLBACK_OOD; no
        #   candidate survives band_filter -> FALLBACK_NO_CANDIDATES. Each fallback = solid_move.
        # - Teaching path: analyst multipv candidates -> band_filter -> score each with
        #   w_close * closeness_score + w_target * target_fn -> argmax.
        # - Record every decision (with candidates and scores) for later analysis.
        # TODO: implement
        raise NotImplementedError

    def _record(self, d: TeacherDecision) -> TeacherDecision:
        """Append to self.decisions and return d."""
        # TODO: implement
        raise NotImplementedError

    def choose_move(self, board: chess.Board) -> chess.Move:      # MovePolicy
        """MovePolicy entry point: the move of decide()."""
        # TODO: implement
        raise NotImplementedError

## Self-check

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - band_filter with candidates e2e4 +300, d2d4 +220, a2a3 +100, h2h4 +250 (mate -3), max_drop 120
#   -> [e2e4, d2d4] (a2a3 too far, h2h4 gets mated).
# - closeness_score: 100 -> 1.0, -150 -> 1.0, 450 -> 0.0 (with the default band).
# - No history -> FALLBACK_LOW_CONFIDENCE and a legal move.
# - Confident model (8 synthetic games, see player_model's `_synthetic`), target_fn = lambda *_: 0:
#   in "4k3/8/8/3q4/8/8/8/3QK3 w - - 0 1" with the default max drop it must still take the queen
#   (d1d5, reason TEACHING); with max_eval_drop_cp 2000 and multipv 50, closeness wins: it does not
#   take the queen and the chosen move scores 1.0.
# - 12 live blunders in the current game -> FALLBACK_OOD.
# Example shape of a test:
# pol = TeacherPolicy(MaterialEngine(depth=1), MaterialEngine(depth=1), PlayerModel("new"))
# assert pol.decide(chess.Board()).reason == FALLBACK_LOW_CONFIDENCE
print("Self-check: not written yet (see plan above)")